In [18]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

In [19]:
df = pd.read_csv('C:/Users/nguye/SteamCrawler/Preprocessing/data.csv')


In [20]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 29906 entries, 0 to 29905
Data columns (total 17 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Appid              29906 non-null  int64 
 1   Name               29906 non-null  object
 2   Type               29906 non-null  object
 3   ReleaseDate        29906 non-null  object
 4   Developers         29906 non-null  object
 5   Publishers         29906 non-null  object
 6   Description        29906 non-null  object
 7   price              29906 non-null  object
 8   Thumbnail          29906 non-null  object
 9   Tags               29906 non-null  object
 10  ReviewScore        29906 non-null  int64 
 11  PositiveReview     29906 non-null  int64 
 12  NegativeReview     29906 non-null  int64 
 13  OsRequirement      29906 non-null  object
 14  MemoryRequirement  29906 non-null  object
 15  CpuRequirement     29906 non-null  object
 16  Rank               29906 non-null  int64

In [21]:
df.isnull().sum() 

Appid                0
Name                 0
Type                 0
ReleaseDate          0
Developers           0
Publishers           0
Description          0
price                0
Thumbnail            0
Tags                 0
ReviewScore          0
PositiveReview       0
NegativeReview       0
OsRequirement        0
MemoryRequirement    0
CpuRequirement       0
Rank                 0
dtype: int64

In [22]:
df.isna().sum()

Appid                0
Name                 0
Type                 0
ReleaseDate          0
Developers           0
Publishers           0
Description          0
price                0
Thumbnail            0
Tags                 0
ReviewScore          0
PositiveReview       0
NegativeReview       0
OsRequirement        0
MemoryRequirement    0
CpuRequirement       0
Rank                 0
dtype: int64

In [23]:
#xử lí dữ liệu price, đây đang là object, cần chuyển về float 
df['price'] = df['price'].str.replace('$', '').astype(float)

In [24]:
display(df['OsRequirement'].value_counts())

OsRequirement
Windows 10                                3349
No requirement                            2923
Windows 7                                 2303
Windows 10 64-bit                          626
Windows XP                                 514
                                          ... 
32 bits                                      1
Windows 10 [64-bit]                          1
Windows® 10 Home 64 Bit (version 21H1)       1
XP XP                                        1
Windows 10 64-bit (1903+) / Windows 11       1
Name: count, Length: 4027, dtype: int64

In [41]:
#có tới 4027 giá trị khác nhau, nên sẽ rất khó để mã hóa, ta sẽ chia Os thành 5 nhóm chính 
def extract_os(x):
    x = str(x).lower()
    if 'no requirement' in x:
        return ['no requirement']
    os_list = []
    if 'windows' in x or 'win' in x:
        os_list.append('windows')
    if 'mac' in x or 'os x' in x:
        os_list.append('mac')
    if 'linux' in x or 'ubuntu' in x:
        os_list.append('linux')
    if not os_list:
        os_list.append('unknown')   
    return os_list
df['OsNew'] = df['OsRequirement'].apply(extract_os)

In [42]:
display(df['OsNew'].value_counts())

OsNew
[windows]                26190
[no requirement]          2923
[unknown]                  774
[windows, mac]              15
[windows, linux]             2
[windows, mac, linux]        1
[mac]                        1
Name: count, dtype: int64

In [45]:
df[df['OsNew'].apply(lambda x: 'unknown' in x)]['OsRequirement'].value_counts().head(20)

OsRequirement
10                                                  128
7                                                    71
Requires a 64-bit processor and operating system     48
XP                                                   42
7 / 8 / 8.1 / 10                                     33
Any                                                  26
Vista                                                15
XP, Vista, 7, 8, 10                                  12
7/8.1/10                                             12
7+                                                    9
7,8,10,11                                             9
XP, Vista, 7, 8                                       8
XP/Vista/7/8/10                                       8
7/8/10                                                7
7 / 8 / 10                                            7
Vista, 7, 8, 10                                       6
10+                                                   6
8                                 

In [49]:
#có thể thấy các giá trị này do viết quá tắt gây ra, nhưng có thể chia thành các nhóm 
# với any tương đương 'no requirement'
# Với requires a 64-bit processor, có thể coi là 'no requirement' vì đa số máy hiện nay đều có bộ xử lý 64-bit, nên ta sẽ gán nó vào nhóm 'no requirement'
# Với các máy còn lại thì có thể gán vào nhóm 'windows' vì đa số các máy này đều chạy windows, vì vista, xp, 7, 8, 10, 11 đều là hệ điều hành windows
# còn 1 trường hợp đặc biệt là có thể sử dụng trên 3 hdh chính là win, mac, linux thì cũng sẽ tương đương với 'no requirement' 
df['OsNew'] = df['OsNew'].apply(lambda x: ['no requirement'] if 'any' in x or 'requires a 64-bit processor' in x else (['windows'] if 'unknown' in x else x))
df['OsNew'] = df['OsNew'].apply(lambda x: ['no requirement'] if 'windows' in x and 'mac' in x and 'linux' in x else x)

In [50]:
display(df['OsNew'].value_counts())

OsNew
[windows]           26964
[no requirement]     2924
[windows, mac]         15
[windows, linux]        2
[mac]                   1
Name: count, dtype: int64

In [51]:
# Bây giờ ta đã tạo ra nhóm dễ xử lí hơn với 5 nhóm chính, giờ ta sẽ thay vào cột os requirement để tiện cho việc mã hóa sau này
df['OsRequirement'] = df['OsNew']
df.drop(columns=['OsNew'], inplace=True)

In [53]:
display(df['OsRequirement'].value_counts())

OsRequirement
[windows]           26964
[no requirement]     2924
[windows, mac]         15
[windows, linux]        2
[mac]                   1
Name: count, dtype: int64

In [55]:
# Tiếp tục là xử lí về ram .
display(df['MemoryRequirement'].value_counts())

MemoryRequirement
4 GB RAM                                                     7427
8 GB RAM                                                     5867
2 GB RAM                                                     4303
No requirement                                               3970
1 GB RAM                                                     2334
                                                             ... 
1GB (XP), 2GB (Vista), 2GB (Windows 7) GB RAM                   1
69 MB RAM                                                       1
8128 MB RAM                                                     1
512Mb RAM for Windows XP (1GB for Windows Vista & 7/8/10)       1
4GB of RAM                                                      1
Name: count, Length: 400, dtype: int64